In [27]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.7-py3-none-any.whl size=53345 sha256=a5cd97c16f376c03ffc56355a03062bb4ae21ebf6d1bc454ce0036eb5dcbd04e
  Stored in directory: /root/.cache/pip/wheels/05/bd/e2/0c0eecf3c26b6cdb231b5656d53905bddc8090d69619d57e6b
Successfully built grad-cam


# Baseline model for: Trustworthy Deep Learning for Chest X-ray Disease Detection
## Stage 1 - Baseline Establishment
 
Architecture : DenseNet121 (ImageNet-pretrained, timm), 4-class head

Dataset      : COVID-19 Radiography Database (Kaggle) - COVID / Normal / Lung_Opacity / Viral Pneumonia

Split        : 70/15/15 stratified, fixed seed

Preprocessing: 224x224, 3-channel, ImageNet normalization

Augmentation : random rotation, horizontal flip, brightness/contrast jitter, random crop (train only)

Optimizer    : AdamW (SGD-momentum / Adam selectable for the hyperparameter sweep)

Loss         : Cross-entropy with class weighting

Training     : two-phase transfer learning (frozen backbone -> fine-tune final dense blocks)
              + early stopping on validation loss

Evaluation   : accuracy, precision, recall, F1 (per-class + macro), ROC-AUC, confusion matrix
 
Run on Google Colab / Kaggle (free GPU). Example:
    python baseline_densenet121.py --data_dir /kaggle/input/covid19-radiography-database \
        --output_dir ./runs/densenet121_baseline


In [28]:
import argparse
import copy
import json
import os
import random
from pathlib import Path
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score, accuracy_score

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
 
try:
    import timm
except ImportError as e:
    raise ImportError(
        "This script requires `timm`. Install with: pip install timm --break-system-packages"
    ) from e

## Reproducibility

In [29]:
def set_seed(seed: int = 42):
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Data

In [30]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [31]:
def build_transforms(img_size: int = 224):
  train_tf = transforms.Compose(
    [
      transforms.Resize((img_size, img_size)),
      transforms.RandomCrop(img_size, padding=8, padding_mode='reflect'),
      transforms.RandomHorizontalFlip(p=0.5),
      transforms.RandomRotation(degrees=10),
      transforms.ColorJitter(brightness=0.2, contrast=0.2),
      transforms.Grayscale(num_output_channels=3),
      transforms.ToTensor(),
      transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ]
  )

  eval_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
  ])
  return train_tf, eval_tf

In [32]:
class TransformSubset(Dataset):
  """Wraps a Subset of an ImageFolder so train/val/test can each use a different transform."""

  def __init__(self, subset: Subset, transform):
    self.subset = subset
    self.transform = transform

  def __len__(self):
    return len(self.subset)

  def __getitem__(self, idx):
    img, label = self.subset[idx]
    if self.transform is not None:
      img = self.transform(img)
    return img, label

In [33]:
def stratified_split(dataset: ImageFolder, seed: int = 42):
  """70/15/15 stratified split of dataset into train, val, and test subsets."""

  targets = np.array(dataset.targets)
  indices = np.arange(len(dataset))

  train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, stratify=targets, random_state=seed
  )

  val_idx, test_idx = train_test_split(
    temp_idx,
    test_size= 0.50,
    stratify=targets[temp_idx],
    random_state = seed,
  )

  return train_idx, val_idx, test_idx

In [34]:
def build_dataloaders(data_dir: str, img_size: int, batch_size: int, seed: int, num_workers: int = 4):
  # ImageFolder expects: data_dir/<class_name>/*.png
  # Load once without transform so PIL images can be transformed differently per split.
  def only_images_folder(path):
    p = Path(path)
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif"}
    return p.parent.name.lower() == "images" and p.suffix.lower() in valid_exts
  base_dataset = ImageFolder(root=data_dir, is_valid_file=only_images_folder)

  train_idx, val_idx, test_idx = stratified_split(base_dataset, seed=seed)
  train_tf, eval_tf = build_transforms(img_size)

  train_ds = TransformSubset(Subset(base_dataset, train_idx), train_tf)
  val_ds = TransformSubset(Subset(base_dataset, val_idx), eval_tf)
  test_ds = TransformSubset(Subset(base_dataset, test_idx), eval_tf)

  pin_memory = torch.cuda.is_available()
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

  class_names = base_dataset.classes
  train_targets = np.array(base_dataset.targets)[train_idx]
  datasets = {"train": train_ds, "val": val_ds, "test": test_ds}
  return train_loader, val_loader, test_loader, class_names, train_targets, datasets

In [35]:
def compute_class_weights(train_targets: np.ndarray, num_classes: int) -> torch.Tensor:
  counts = np.bincount(train_targets, minlength=num_classes).astype(np.float32)
  counts[counts == 0] = 1.0  # avoid div-by-zero
  weights = counts.sum() / (num_classes * counts)
  return torch.tensor(weights, dtype=torch.float32)

## Model

In [36]:
def build_model(num_classes: int = 4) -> nn.Module:
    model = timm.create_model("densenet121", pretrained=True, num_classes=num_classes)
    return model

In [37]:
def freeze_backbone(model: nn.Module):
    """Phase 1: freeze everything except the final classifier head."""
    for name, param in model.named_parameters():
        if "classifier" in name or "fc" in name:  # timm densenet head is named 'classifier'
            param.requires_grad = True
        else:
            param.requires_grad = False

In [38]:
def unfreeze_final_blocks(model: nn.Module, num_dense_blocks_to_unfreeze: int = 1):
    """Phase 2: unfreeze the classifier + the last N dense blocks for end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if "classifier" in name:
            param.requires_grad = True
    # timm densenet121 feature blocks are named features.denseblock1..4 / features.norm5
    unfreeze_names = ["features.norm5"] + [
        f"features.denseblock{4 - i}" for i in range(num_dense_blocks_to_unfreeze)
    ] + [
        f"features.transition{3 - i}" for i in range(num_dense_blocks_to_unfreeze)
    ]
    for name, param in model.named_parameters():
        if any(name.startswith(u) for u in unfreeze_names):
            param.requires_grad = True

In [39]:
def build_optimizer(model, name: str, lr: float, weight_decay: float):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

In [40]:
def build_scheduler(optimizer, name: str, epochs: int):
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.1)
    elif name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    elif name == "none":
        return None
    raise ValueError(f"Unknown scheduler: {name}")

## Training / evaluation loops

In [41]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool, desc: str = ""):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
 
    non_blocking = device.type == "cuda"
    context = torch.enable_grad() if train else torch.no_grad()
    progress = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    with context:
        for images, labels in progress:
            images = images.to(device, non_blocking=non_blocking)
            labels = labels.to(device, non_blocking=non_blocking)
            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
 
            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size
            progress.set_postfix(loss=f"{total_loss / total:.4f}", acc=f"{correct / total:.4f}")
 
    return total_loss / total, correct / total

In [42]:
def train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, epochs, patience, phase_name, output_dir,
):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = []
 
    for epoch in range(1, epochs + 1):
        train_desc = f"{phase_name} epoch {epoch}/{epochs} train"
        val_desc = f"{phase_name} epoch {epoch}/{epochs} val"
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, desc=train_desc
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, desc=val_desc
        )
 
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
 
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{phase_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break
 
    model.load_state_dict(best_state)
    with open(Path(output_dir) / f"{phase_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model

In [43]:
@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir, evaluation_type):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
 
    non_blocking = device.type == "cuda"
    for images, labels in loader:
        images = images.to(device, non_blocking=non_blocking)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
 
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
 
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
 
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)
 
    try:
        auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(all_labels, all_probs, multi_class="ovr", average=None)
    except ValueError:
        auc_macro, auc_per_class = None, None
 
    results = {
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "roc_auc_macro": auc_macro,
        "roc_auc_per_class": auc_per_class.tolist() if auc_per_class is not None else None,
        "class_names": class_names,
    }
 
    with open(Path(output_dir) / f"{evaluation_type}_test_results.json", "w") as f:
        json.dump(results, f, indent=2)
 
    print("\n=== Test set performance ===")
    print(f"Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print("Confusion matrix:\n", cm)
    if auc_macro is not None:
        print(f"Macro ROC-AUC: {auc_macro:.4f}")

    # --- Save confusion matrix as CSV ---
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(Path(output_dir) / f"{evaluation_type}_confusion_matrix.csv")

    # --- Save one-row summary CSV (accuracy, macro/weighted f1 & recall, ROC-AUC) ---
    summary = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_precision": report["macro avg"]["precision"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "weighted_recall": report["weighted avg"]["recall"],
        "roc_auc_macro": auc_macro,
    }
    pd.DataFrame([summary]).to_csv(Path(output_dir) / f"{evaluation_type}_summary_metrics.csv", index=False)
    
 
    return results

In [44]:
parser = argparse.ArgumentParser(description="DenseNet121 baseline - Stage 1")
parser.add_argument("--data_dir", type=str,
                    default=r"/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset",
                    help="Path to dataset root with one subfolder per class")
parser.add_argument("--output_dir", type=str, default="./runs/densenet121_baseline")
parser.add_argument("--img_size", type=int, default=224)
parser.add_argument("--batch_size", type=int, default=32)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--num_workers", type=int, default=4)
parser.add_argument("--device", type=str, default="auto", choices=["auto", "cuda", "cpu"],
                     help="'auto' picks CUDA if available, else CPU. Force 'cpu' to sanity-check "
                          "the pipeline locally before running full training on a GPU box.")

# Phase 1 (frozen backbone, train head)
parser.add_argument("--phase1_epochs", type=int, default=15)
parser.add_argument("--phase1_lr", type=float, default=1e-3)

# Phase 2 (fine-tune final dense blocks)
parser.add_argument("--phase2_epochs", type=int, default=40)
parser.add_argument("--phase2_lr", type=float, default=1e-5)
parser.add_argument("--unfreeze_blocks", type=int, default=1)

# Hyperparameter-sweep knobs
parser.add_argument("--optimizer", type=str, default="adamw", choices=["adamw", "adam", "sgd"])
parser.add_argument("--weight_decay", type=float, default=1e-4)
parser.add_argument("--scheduler", type=str, default="cosine", choices=["cosine", "step", "plateau", "none"])
parser.add_argument("--patience", type=int, default=5)

_StoreAction(option_strings=['--patience'], dest='patience', nargs=None, const=None, default=5, type=<class 'int'>, choices=None, required=False, help=None, metavar=None)

In [45]:
args, _ = parser.parse_known_args()

Path(args.output_dir).mkdir(parents=True, exist_ok=True)
set_seed(args.seed)

In [46]:
if args.device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("--device cuda was requested but no CUDA GPU is available.")
if args.device == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(args.device)

print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(device)}")
    print(f"  CUDA version: {torch.version.cuda}")
    total_mem_gb = torch.cuda.get_device_properties(device).total_memory / (1024 ** 3)
    print(f"  GPU memory: {total_mem_gb:.1f} GB")
else:
    print("  Running on CPU - training DenseNet121 here will be much slower; "
          "use --device cpu only for a quick smoke test with a small --batch_size.")


Using device: cuda
  GPU: Tesla T4
  CUDA version: 12.8
  GPU memory: 14.6 GB


In [47]:
# ---- Data ----
train_loader, val_loader, test_loader, class_names, train_targets, datasets = build_dataloaders(
    args.data_dir, args.img_size, args.batch_size, args.seed, args.num_workers
)
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")
print(f"Train/Val/Test sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}")

class_weights = compute_class_weights(train_targets, num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

Classes (4): ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Train/Val/Test sizes: 14815/3175/3175


## Cache preprocessed images (optional)

Writes the fully preprocessed (resize + grayscale + normalize) images to disk as PNGs, one folder per split/class, so the expensive PIL decode/resize doesn't have to be repeated on every epoch or run.

Note: the train split's random augmentation (crop/flip/rotation/jitter) is intentionally **not** baked into this cache — that transform is meant to produce a different result each epoch, so caching one fixed draw of it would silently remove that variability. The train images below are cached with the same deterministic transform used for val/test.

In [48]:
def cache_preprocessed_images(dataset, class_names, split_name: str, output_dir, mean=IMAGENET_MEAN, std=IMAGENET_STD):
  """Run every sample in `dataset` through its transform once and save the result as a PNG.

  Denormalizes back to [0, 1] before saving so the cached files are viewable images
  rather than raw normalized tensors.
  """
  split_dir = Path(output_dir) / split_name
  for cls in class_names:
    (split_dir / cls).mkdir(parents=True, exist_ok=True)

  mean_t = torch.tensor(mean).view(3, 1, 1)
  std_t = torch.tensor(std).view(3, 1, 1)
  to_pil = transforms.ToPILImage()
  counts = {cls: 0 for cls in class_names}

  for img_tensor, label in tqdm(dataset, desc=f"Caching {split_name}", dynamic_ncols=True):
    cls = class_names[label]
    denorm = (img_tensor * std_t + mean_t).clamp(0, 1)
    to_pil(denorm).save(split_dir / cls / f"{counts[cls]:05d}.png")
    counts[cls] += 1

  print(f"[{split_name}] cached {sum(counts.values())} images to {split_dir}")
  return counts

In [58]:
# Deterministic (eval-style) view of the train split — same underlying samples, no random augmentation.
train_ds_deterministic = TransformSubset(datasets["train"].subset, datasets["val"].transform)

cache_dir = Path(args.output_dir) / "preprocessed_images"
cache_preprocessed_images(train_ds_deterministic, class_names, "train", cache_dir)
cache_preprocessed_images(datasets["val"], class_names, "val", cache_dir)
cache_preprocessed_images(datasets["test"], class_names, "test", cache_dir)

Caching train: 100%|██████████| 14815/14815 [05:02<00:00, 49.01it/s]


[train] cached 14815 images to runs/densenet121_baseline/preprocessed_images/train


Caching val: 100%|██████████| 3175/3175 [01:06<00:00, 47.95it/s]


[val] cached 3175 images to runs/densenet121_baseline/preprocessed_images/val


Caching test: 100%|██████████| 3175/3175 [01:05<00:00, 48.33it/s]

[test] cached 3175 images to runs/densenet121_baseline/preprocessed_images/test


{'COVID': 542, 'Lung_Opacity': 902, 'Normal': 1529, 'Viral Pneumonia': 202}

In [59]:
model = build_model(num_classes=num_classes).to(device)

In [60]:
freeze_backbone(model)
opt1 = build_optimizer(model, args.optimizer, args.phase1_lr, args.weight_decay)
sched1 = build_scheduler(opt1, args.scheduler, args.phase1_epochs)
model = train_phase(
    model, train_loader, val_loader, criterion, opt1, sched1,
    device, args.phase1_epochs, args.patience, "phase1_frozen", args.output_dir,
)

[phase1_frozen] epoch 1/15 train_loss=0.6505 train_acc=0.7252 val_loss=0.4700 val_acc=0.8170


[phase1_frozen] epoch 2/15 train_loss=0.4705 train_acc=0.7955 val_loss=0.4048 val_acc=0.8306


[phase1_frozen] epoch 3/15 train_loss=0.4374 train_acc=0.8061 val_loss=0.4005 val_acc=0.8438


[phase1_frozen] epoch 4/15 train_loss=0.4103 train_acc=0.8250 val_loss=0.3967 val_acc=0.8324


[phase1_frozen] epoch 5/15 train_loss=0.3940 train_acc=0.8199 val_loss=0.3669 val_acc=0.8476


...

[phase1_frozen] epoch 11/15 train_loss=0.3561 train_acc=0.8430 val_loss=0.3348 val_acc=0.8529


[phase1_frozen] epoch 12/15 train_loss=0.3577 train_acc=0.8402 val_loss=0.3333 val_acc=0.8583


[phase1_frozen] epoch 13/15 train_loss=0.3565 train_acc=0.8403 val_loss=0.3332 val_acc=0.8580


[phase1_frozen] epoch 14/15 train_loss=0.3471 train_acc=0.8468 val_loss=0.3325 val_acc=0.8570


[phase1_frozen] epoch 15/15 train_loss=0.3528 train_acc=0.8416 val_loss=0.3322 val_acc=0.8570


In [61]:
evaluate(model, test_loader, class_names, device, args.output_dir, "frozen")


=== Test set performance ===
Test accuracy: 0.8617
                 precision    recall  f1-score   support

          COVID     0.7961    0.8210    0.8084       542
   Lung_Opacity     0.8414    0.8647    0.8529       902
         Normal     0.9058    0.8620    0.8834      1529
Viral Pneumonia     0.8248    0.9554    0.8853       202

       accuracy                         0.8617      3175
      macro avg     0.8420    0.8758    0.8575      3175
   weighted avg     0.8636    0.8617    0.8620      3175

Confusion matrix:
 [[ 445   40   51    6]
 [  43  780   78    1]
 [  71  106 1318   34]
 [   0    1    8  193]]
Macro ROC-AUC: 0.9719


{'classification_report': {'COVID': {'precision': 0.7960644007155635,
   'recall': 0.8210332103321033,
   'f1-score': 0.8083560399636693,
   'support': 542.0},
  'Lung_Opacity': {'precision': 0.8414239482200647,
   'recall': 0.8647450110864745,
   'f1-score': 0.8529250956806999,
   'support': 902.0},
  'Normal': {'precision': 0.9058419243986254,
   'recall': 0.8620013080444735,
   'f1-score': 0.8833780160857909,
   'support': 1529.0},
  'Viral Pneumonia': {'precision': 0.8247863247863247,
   'recall': 0.9554455445544554,
   'f1-score': 0.8853211009174312,
   'support': 202.0},
  'accuracy': 0.861732283464567,
  'macro avg': {'precision': 0.8420291495301446,
   'recall': 0.8758062685043767,
   'f1-score': 0.8574950631618978,
   'support': 3175.0},
  'weighted avg': {'precision': 0.8636442351164314,
   'recall': 0.861732283464567,
   'f1-score': 0.8620432311637151,
   'support': 3175.0}},
 'confusion_matrix': [[445, 40, 51, 6],
  [43, 780, 78, 1],
  [71, 106, 1318, 34],
  [0, 1, 8, 193]]

In [62]:
unfreeze_final_blocks(model, num_dense_blocks_to_unfreeze=args.unfreeze_blocks)
opt2 = build_optimizer(model, args.optimizer, args.phase2_lr, args.weight_decay)
sched2 = build_scheduler(opt2, args.scheduler, args.phase2_epochs)
model = train_phase(
    model, train_loader, val_loader, criterion, opt2, sched2,
    device, args.phase2_epochs, args.patience, "phase2_finetune", args.output_dir,
)

[phase2_finetune] epoch 1/40 train_loss=0.3479 train_acc=0.8499 val_loss=0.3154 val_acc=0.8633


[phase2_finetune] epoch 2/40 train_loss=0.3188 train_acc=0.8566 val_loss=0.3015 val_acc=0.8724


[phase2_finetune] epoch 3/40 train_loss=0.3135 train_acc=0.8610 val_loss=0.2921 val_acc=0.8759


[phase2_finetune] epoch 4/40 train_loss=0.3049 train_acc=0.8593 val_loss=0.2769 val_acc=0.8769


[phase2_finetune] epoch 5/40 train_loss=0.2907 train_acc=0.8677 val_loss=0.2718 val_acc=0.8844


...

[phase2_finetune] epoch 35/40 train_loss=0.1934 train_acc=0.9124 val_loss=0.2061 val_acc=0.9134


[phase2_finetune] epoch 36/40 train_loss=0.1958 train_acc=0.9111 val_loss=0.2039 val_acc=0.9150


[phase2_finetune] epoch 37/40 train_loss=0.1960 train_acc=0.9112 val_loss=0.2026 val_acc=0.9068


[phase2_finetune] epoch 38/40 train_loss=0.1958 train_acc=0.9086 val_loss=0.1984 val_acc=0.9099


[phase2_finetune] epoch 39/40 train_loss=0.1911 train_acc=0.9122 val_loss=0.2030 val_acc=0.9112
[phase2_finetune] Early stopping at epoch 39 (no improvement for 5 epochs).


In [63]:
ckpt_path = Path(args.output_dir) / "densenet121_baseline.pt"
torch.save({"model_state_dict": model.state_dict(), "class_names": class_names}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

Saved checkpoint to runs/densenet121_baseline/densenet121_baseline.pt


In [64]:
# ---- Evaluate on held-out test set ----
evaluate(model, test_loader, class_names, device, args.output_dir, "unfreeze_finetune")


=== Test set performance ===
Test accuracy: 0.9169
                 precision    recall  f1-score   support

          COVID     0.8920    0.9446    0.9176       542
   Lung_Opacity     0.9027    0.8947    0.8987       902
         Normal     0.9358    0.9150    0.9253      1529
Viral Pneumonia     0.9104    0.9554    0.9324       202

       accuracy                         0.9169      3175
      macro avg     0.9102    0.9274    0.9185      3175
   weighted avg     0.9173    0.9169    0.9168      3175

Confusion matrix:
 [[ 512   12   15    3]
 [  22  807   73    0]
 [  39   75 1399   16]
 [   1    0    8  193]]
Macro ROC-AUC: 0.9873


{'classification_report': {'COVID': {'precision': 0.89198606271777,
   'recall': 0.9446494464944649,
   'f1-score': 0.9175627240143369,
   'support': 542.0},
  'Lung_Opacity': {'precision': 0.9026845637583892,
   'recall': 0.8946784922394678,
   'f1-score': 0.8986636971046771,
   'support': 902.0},
  'Normal': {'precision': 0.9357859531772575,
   'recall': 0.9149771092217135,
   'f1-score': 0.9252645502645502,
   'support': 1529.0},
  'Viral Pneumonia': {'precision': 0.910377358490566,
   'recall': 0.9554455445544554,
   'f1-score': 0.9323671497584541,
   'support': 202.0},
  'accuracy': 0.9168503937007874,
  'macro avg': {'precision': 0.9102084845359958,
   'recall': 0.9274376481275254,
   'f1-score': 0.9184645302855047,
   'support': 3175.0},
  'weighted avg': {'precision': 0.9172884634098329,
   'recall': 0.9168503937007874,
   'f1-score': 0.9168445079716202,
   'support': 3175.0}},
 'confusion_matrix': [[512, 12, 15, 3],
  [22, 807, 73, 0],
  [39, 75, 1399, 16],
  [1, 0, 8, 193]],


# Define ViT-Base Model

In [51]:
def build_vit_model(num_classes: int = 4) -> nn.Module:
    model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
    return model

def freeze_vit_backbone(model: nn.Module):
    for name, param in model.named_parameters():
        if "head" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

def unfreeze_vit_blocks(model: nn.Module, num_blocks_to_unfreeze: int = 2):
    for param in model.parameters():
        param.requires_grad = False
        
    for name, param in model.named_parameters():
        if "head" in name or "norm" in name:
            param.requires_grad = True
            
    total_blocks = len(model.blocks)
    target_blocks = [f"blocks.{total_blocks - 1 - i}" for i in range(num_blocks_to_unfreeze)]
    
    for name, param in model.named_parameters():
        if any(b in name for b in target_blocks):
            param.requires_grad = True

# Run ViT-Base Phase 1 (Frozen Backbone)

In [ ]:
"""# Setup ViT-Base
vit_output_dir = "./runs/vit_base_baseline"
Path(vit_output_dir).mkdir(parents=True, exist_ok=True)

# Build ViT-Base
vit_model = build_vit_model(num_classes=num_classes).to(device)
freeze_vit_backbone(vit_model)

# Training
opt1_vit = build_optimizer(vit_model, args.optimizer, args.phase1_lr, args.weight_decay)
sched1_vit = build_scheduler(opt1_vit, args.scheduler, args.phase1_epochs)

vit_model = train_phase(
    vit_model, train_loader, val_loader, criterion, opt1_vit, sched1_vit,
    device, args.phase1_epochs, args.patience, "phase1_vit_frozen", vit_output_dir
)

# Evaluate
evaluate(vit_model, test_loader, class_names, device, vit_output_dir, "vit_frozen")"""

# Run ViT-Base Phase 2 (Fine-tuning)

In [ ]:
"""# Phase 2 Fine-Tuning
unfreeze_vit_blocks(vit_model, num_blocks_to_unfreeze=2)

opt2_vit = build_optimizer(vit_model, args.optimizer, args.phase2_lr, args.weight_decay)
sched2_vit = build_scheduler(opt2_vit, args.scheduler, args.phase2_epochs)

vit_model = train_phase(
    vit_model, train_loader, val_loader, criterion, opt2_vit, sched2_vit,
    device, args.phase2_epochs, args.patience, "phase2_vit_finetune", vit_output_dir
)

vit_ckpt_path = Path(vit_output_dir) / "vit_base_baseline.pt"
torch.save({"model_state_dict": vit_model.state_dict(), "class_names": class_names}, vit_ckpt_path)
print(f"Saved ViT baseline checkpoint to {vit_ckpt_path}")

# Final ViT Test Evaluation
evaluate(vit_model, test_loader, class_names, device, vit_output_dir, "vit_unfreeze_finetune")"""

In [56]:
import torch
from pathlib import Path

# Build ViT architecture
vit_model = build_vit_model(num_classes=num_classes).to(device)

# Paste your copied file path here (include .pt extension)
checkpoint_path = "/kaggle/input/datasets/uom230651c/vit-base-baseline/vit_base_baseline.pt"

# Load weights
checkpoint = torch.load(checkpoint_path, map_location=device)
vit_model.load_state_dict(checkpoint["model_state_dict"])
vit_model.eval()

print("ViT-Base baseline weights loaded successfully!")

ViT-Base baseline weights loaded successfully!


In [ ]:
print(f"Model architecture type: {type(vit_model).__name__}")
print(f"Number of parameters: {sum(p.numel() for p in vit_model.parameters()):,}")

dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    dummy_output = vit_model(dummy_input)

print(f"Output batch shape: {dummy_output.shape}")  # Should be torch.Size([2, 4])
print("Uploaded checkpoint is valid and producing 4-class logits!")

In [ ]:
!pip install grad-cam -q

import torch
import torch.nn as nn
import torch.nn.functional as F

class CandidateCLoss(nn.Module):
    def __init__(self, model, lambda_suppress: float = 0.5):
        super().__init__()
        self.model = model
        self.lambda_suppress = lambda_suppress
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights)
        self.current_images = None
        self.current_masks = None

    def set_current_batch(self, images, lung_masks=None):
        self.current_images = images
        self.current_masks = lung_masks

    def forward(self, outputs, labels, images=None, lung_masks=None):
        loss_ce = self.ce_loss(outputs, labels)
        
        imgs = images if images is not None else self.current_images
        masks = lung_masks if lung_masks is not None else self.current_masks
        
        if imgs is None or masks is None or self.lambda_suppress == 0:
            return loss_ce
            
        features = self.model.features(imgs)
        activations = F.relu(features)
        
        masks_resized = F.interpolate(masks, size=activations.shape[2:], mode='bilinear', align_corners=False)
        background_masks = 1.0 - masks_resized
        
        out_of_bounds_energy = (activations * background_masks).mean()
        return loss_ce + (self.lambda_suppress * out_of_bounds_energy)

print("Task T20: Candidate (c) Loss updated for standard run_epoch compatibility!")

# Candidate (c) Grad-CAM Shortcut-Suppression Loss on DenseNet12

In [65]:
!pip install grad-cam -q

import torch
import torch.nn as nn
import torch.nn.functional as F

class CandidateCLoss(nn.Module):
    def __init__(self, model, lambda_suppress: float = 0.5):
        super().__init__()
        self.model = model
        self.lambda_suppress = lambda_suppress
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights)
        self.current_images = None
        self.current_masks = None

    def set_current_batch(self, images, lung_masks=None):
        self.current_images = images
        self.current_masks = lung_masks

    def forward(self, outputs, labels, images=None, lung_masks=None):
        loss_ce = self.ce_loss(outputs, labels)
        
        imgs = images if images is not None else self.current_images
        masks = lung_masks if lung_masks is not None else self.current_masks
        
        if imgs is None or masks is None or self.lambda_suppress == 0:
            return loss_ce
            
        features = self.model.features(imgs)
        activations = F.relu(features)
        
        masks_resized = F.interpolate(masks, size=activations.shape[2:], mode='bilinear', align_corners=False)
        background_masks = 1.0 - masks_resized  # 1 on background/artifacts, 0 inside lungs
        
        out_of_bounds_energy = (activations * background_masks).mean()
        return loss_ce + (self.lambda_suppress * out_of_bounds_energy)

print("Task T20: Candidate (c) Shortcut-Suppression Loss defined!")

Task T20: Candidate (c) Shortcut-Suppression Loss defined!


In [66]:
# Exploratory Regularized Training Trial

import timm
from pathlib import Path

densenet_suppress_dir = "./runs/densenet121_shortcut_suppression"
Path(densenet_suppress_dir).mkdir(parents=True, exist_ok=True)

# Build DenseNet121 model
densenet_model = timm.create_model('densenet121', pretrained=True, num_classes=num_classes).to(device)

# Instantiate Candidate (c) Loss
candidate_c_criterion = CandidateCLoss(densenet_model, lambda_suppress=0.5)

# Setup Optimizer & Scheduler
opt_candidate_c = torch.optim.AdamW(densenet_model.parameters(), lr=1e-4, weight_decay=1e-2)
sched_candidate_c = torch.optim.lr_scheduler.CosineAnnealingLR(opt_candidate_c, T_max=10)

print("Starting Candidate (c) Shortcut-Suppression Fine-Tuning Trial...")

# Train DenseNet121 with Candidate (c) Regularization
densenet_model = train_phase(
    densenet_model, train_loader, val_loader, candidate_c_criterion, opt_candidate_c, sched_candidate_c,
    device, epochs=10, patience=5, phase_name="candidate_c_suppression", output_dir=densenet_suppress_dir
)

# Save Regularized Checkpoint
suppress_ckpt_path = Path(densenet_suppress_dir) / "densenet121_candidate_c.pt"
torch.save({"model_state_dict": densenet_model.state_dict(), "class_names": class_names}, suppress_ckpt_path)
print(f"Saved Candidate (c) regularized checkpoint to {suppress_ckpt_path}")

# Evaluate Test Performance
evaluate(densenet_model, test_loader, class_names, device, densenet_suppress_dir, "candidate_c_test_results")

Starting Candidate (c) Shortcut-Suppression Fine-Tuning Trial...


[candidate_c_suppression] epoch 1/10 train_loss=0.4118 train_acc=0.8349 val_loss=0.2070 val_acc=0.9181


[candidate_c_suppression] epoch 2/10 train_loss=0.1915 train_acc=0.9165 val_loss=0.1487 val_acc=0.9383


[candidate_c_suppression] epoch 3/10 train_loss=0.1597 train_acc=0.9295 val_loss=0.1411 val_acc=0.9254


[candidate_c_suppression] epoch 4/10 train_loss=0.1312 train_acc=0.9426 val_loss=0.1250 val_acc=0.9458


...

[candidate_c_suppression] epoch 7/10 train_loss=0.0759 train_acc=0.9658 val_loss=0.1085 val_acc=0.9531


[candidate_c_suppression] epoch 8/10 train_loss=0.0635 train_acc=0.9711 val_loss=0.1127 val_acc=0.9531


[candidate_c_suppression] epoch 9/10 train_loss=0.0590 train_acc=0.9733 val_loss=0.1104 val_acc=0.9546


[candidate_c_suppression] epoch 10/10 train_loss=0.0535 train_acc=0.9773 val_loss=0.1033 val_acc=0.9562
Saved Candidate (c) regularized checkpoint to runs/densenet121_shortcut_suppression/densenet121_candidate_c.pt

=== Test set performance ===
Test accuracy: 0.9553
                 precision    recall  f1-score   support

          COVID     0.9853    0.9871    0.9862       542
   Lung_Opacity     0.9477    0.9246    0.9360       902
         Normal     0.9495    0.9601    0.9548      1529
Viral Pneumonia     0.9515    0.9703    0.9608       202

       accuracy                         0.9553      3175
      macro avg     0.9585    0.9605    0.9594      3175
   weighted avg     0.9552    0.9553    0.9552      3175

Confusion matrix:
 [[ 535    1    5    1]
 [   1  834   67    0]
 [   7   45 1468    9]
 [   0    0    6  196]]
Macro ROC-AUC: 0.9946


{'classification_report': {'COVID': {'precision': 0.9852670349907919,
   'recall': 0.9870848708487084,
   'f1-score': 0.9861751152073732,
   'support': 542.0},
  'Lung_Opacity': {'precision': 0.9477272727272728,
   'recall': 0.9246119733924612,
   'f1-score': 0.936026936026936,
   'support': 902.0},
  'Normal': {'precision': 0.9495472186287193,
   'recall': 0.9601046435578809,
   'f1-score': 0.9547967479674797,
   'support': 1529.0},
  'Viral Pneumonia': {'precision': 0.9514563106796117,
   'recall': 0.9702970297029703,
   'f1-score': 0.9607843137254902,
   'support': 202.0},
  'accuracy': 0.955275590551181,
  'macro avg': {'precision': 0.9584994592565989,
   'recall': 0.9605246293755052,
   'f1-score': 0.9594457782318198,
   'support': 3175.0},
  'weighted avg': {'precision': 0.9552493244112134,
   'recall': 0.955275590551181,
   'f1-score': 0.9552018481113443,
   'support': 3175.0}},
 'confusion_matrix': [[535, 1, 5, 1],
  [1, 834, 67, 0],
  [7, 45, 1468, 9],
  [0, 0, 6, 196]],
 'roc